In [ ]:
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline


def train_fraud_detector(dataset_path: str):
  # 1. Load the dataset
  print(f'Loading dataset from {dataset_path}...')
  df = pd.read_csv(dataset_path)

  # Validate required columns
  if 'text' not in df.columns or 'label' not in df.columns:
    raise ValueError("Dataset must contain 'text' and 'label' columns.")

  # Clean missing values
  df = df.dropna(subset=['text', 'label'])

  X = df['text']
  y = df['label']

  # 2. Train-Test Split (80% Train, 20% Test)
  X_train, X_test, y_train, y_test = train_test_split(
      X, y, test_size=0.20, random_state=42, stratify=y
  )

  # 3. Create Scikit-Learn Pipeline (TF-IDF Vectorizer + Logistic Regression)
  pipeline = Pipeline([
      (
          'tfidf',
          TfidfVectorizer(
              ngram_range=(1, 2),  # Use unigrams and bigrams
              max_features=10000,  # Limit vocabulary size
              lowercase=True,
              strip_accents='unicode',
          ),
      ),
      ('classifier', LogisticRegression(C=1.0, random_state=42, max_iter=1000)),
  ])

  # 4. Train Model
  print('Training model pipeline...')
  pipeline.fit(X_train, y_train)

  # 5. Evaluate Model
  y_pred = pipeline.predict(X_test)
  acc = accuracy_score(y_test, y_pred)

  print('\n' + '=' * 40)
  print(f'MODEL PERFORMANCE (Accuracy: {acc:.4f})')
  print('=' * 40)
  print(
      classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud'])
  )
  print('Confusion Matrix:')
  print(confusion_matrix(y_test, y_pred))

  # 6. Save Trained Model
  model_filename = 'nigerian_fraud_sms_model.pkl'
  joblib.dump(pipeline, model_filename)
  print(f'\nTrained model pipeline saved successfully as "{model_filename}".')

  return pipeline


# --- Inference Helper Function ---
def predict_sms(model, sms_text: str):
  """Predicts whether an incoming SMS is Fraud or Legitimate."""
  prediction = model.predict([sms_text])[0]
  probabilities = model.predict_proba([sms_text])[0]

  label_map = {0: 'Legitimate', 1: 'Fraud'}

  return {
      'text': sms_text,
      'classification': label_map[prediction],
      'fraud_probability': f'{probabilities[1] * 100:.2f}%',
      'legitimate_probability': f'{probabilities[0] * 100:.2f}%',
  }


if __name__ == '__main__':
  # Train and evaluate
  trained_model = train_fraud_detector('nigerian_fraud_sms_dataset.csv')

  # Test sample Nigerian SMS predictions
  print('\n' + '=' * 40)
  print('SAMPLE INFERENCE TEST')
  print('=' * 40)

  sample_messages = [
      (
          'Congratulations! You have been selected for FG Empowerment Grant of'
          ' N50,000. Click http://fg-grant-claim.site to receive funds.'
      ),
      (
          'Glo: Dear Customer, you have used 80% of your daily data bundle.'
          ' Dial *323# to buy a top-up.'
      ),
      (
          'URGENT: Your bank account has been blocked due to missing BVN.'
          ' Call 08030000000 immediately to unblock.'
      ),
  ]

  for msg in sample_messages:
    res = predict_sms(trained_model, msg)
    print(f"SMS: '{res['text']}'")
    print(
        f"Result: {res['classification']} (Fraud Confidence:"
        f" {res['fraud_probability']})\n"
    )